# CRAFT Stage 2: Dense Semantic Reranking

This notebook implements Stage 2 of the CRAFT pipeline: Dense semantic reranking using mini-tables.

**Pipeline Flow:**
1. Load Stage 1 SPLADE results (5,000 candidates per query)
2. Construct mini-tables from top rows per table  
3. Apply dense embeddings (Sentence Transformers for NQ, JINA for OTT-QA)
4. Rerank and select top 100 candidates for Stage 3

In [ ]:
# Import CRAFT modular components
from craft_core import CRAFTConfig, DataLoader, TableProcessor, ResultsManager, setup_logging
from craft_stages import create_dense_reranker, setup_nq_pipeline, setup_ottqa_pipeline

import pandas as pd
import json
import numpy as np
from tqdm import tqdm
import logging

# Setup logging
setup_logging("INFO")
logger = logging.getLogger(__name__)

# Configuration
DATASET = "nq"  # "nq" or "ottqa"

# Initialize CRAFT configuration
config = CRAFTConfig(DATASET)
data_loader = DataLoader()
table_processor = TableProcessor()
results_manager = ResultsManager()

print(f"🧠 CRAFT Stage 2: Dense Semantic Reranking")
print(f"📊 Dataset: {DATASET.upper()}")
print(f"🎯 Target candidates: {config.stage2_candidates}")

# Get pipeline configuration
pipeline_config = setup_nq_pipeline() if DATASET == "nq" else setup_ottqa_pipeline()
model_name = pipeline_config['stage2']

logger.info(f"🤖 Using model: {model_name}")

In [ ]:
# Load Stage 1 results and supporting data
logger.info("📂 Loading Stage 1 results and supporting data...")

# Load Stage 1 SPLADE results
stage1_results_path = config.base_dir / "datasets" / f"corpus_for_2nd_Stage_{DATASET.upper()}_Splade_CRAFT.pkl"
if not stage1_results_path.exists():
    # Try alternative path
    stage1_results_path = config.base_dir / "results" / "stage1" / f"CRAFT_{DATASET.upper()}_Stage1_Results_*.pkl"
    import glob
    matching_files = glob.glob(str(stage1_results_path))
    if matching_files:
        stage1_results_path = matching_files[0]

stage1_results = data_loader.load_pickle(stage1_results_path)
logger.info(f"✅ Loaded Stage 1 results: {len(stage1_results)} queries")

# Load metadata and questions
metadata = data_loader.load_metadata(config.get_path("metadata"))
questions = data_loader.load_questions(config.get_path("questions"))

logger.info(f"📊 Loaded {len(metadata)} tables and {len(questions)} questions")

In [ ]:
# Load row-level data for mini-table construction
logger.info("📋 Loading row-level data for mini-tables...")

# Load top rows per table (from previous stage processing)
top_rows_path = config.base_dir / "datasets" / "corpus_2nd_stage_row_rerank_with_ST_original_q.pkl"
top_rows_per_table = data_loader.load_pickle(top_rows_path)

# Load row data mapping
row_data_path = config.base_dir / "datasets" / f"{DATASET}_row_tables.json"
with open(row_data_path, 'r') as f:
    row_table_data = json.load(f)

# Create row ID to data mapping
rowid_to_data = {entry['Row Number']: entry for entry in row_table_data}

logger.info(f"✅ Loaded top rows for {len(top_rows_per_table)} queries")
logger.info(f"✅ Loaded {len(rowid_to_data)} row entries")

# Initialize dense reranker
logger.info("🤖 Initializing dense reranker...")
reranker = create_dense_reranker(model_name=model_name, device="cpu")
logger.info("✅ Dense reranker ready")

In [ ]:
# Process queries with dense reranking using mini-tables
logger.info(f"🔄 Processing {len(stage1_results)} queries with dense reranking...")

reranked_results = {}
failed_queries = []

for query_id, stage1_candidates in tqdm(stage1_results.items(), desc="Dense reranking"):
    try:
        # Get query information
        if query_id in questions.index.astype(str):
            query_row = questions.loc[questions.index.astype(str) == query_id].iloc[0]
        else:
            # Try to find by different ID mapping
            query_idx = int(query_id) if query_id.isdigit() else None
            if query_idx is not None and query_idx < len(questions):
                query_row = questions.iloc[query_idx]
            else:
                logger.warning(f"Query {query_id} not found in questions")
                failed_queries.append(query_id)
                continue
        
        question = query_row.get('question', query_row.get('text', ''))
        
        # Extract candidate table IDs and scores from Stage 1
        if isinstance(stage1_candidates, dict):
            table_ids = list(stage1_candidates.keys())
        else:
            table_ids = [tid for tid, _ in stage1_candidates]
        
        # Create mini-tables for candidates
        mini_tables = []
        valid_table_ids = []
        
        for table_id in table_ids:
            # Create mini-table using table processor
            mini_table = table_processor.create_mini_table(
                table_id,
                top_rows_per_table.get(query_id, {}),
                rowid_to_data,
                max_rows=config.mini_table_rows
            )
            
            if mini_table:  # Only include if mini-table was successfully created
                # Add table title from metadata
                table_info = metadata[metadata['TableID'] == table_id]
                if not table_info.empty:
                    title = table_info.iloc[0]['Table_Title']
                    mini_table = f"Title: {title}. Content: {mini_table}"
                
                mini_tables.append(mini_table)
                valid_table_ids.append(table_id)
        
        if not mini_tables:
            logger.warning(f"No mini-tables created for query {query_id}")
            failed_queries.append(query_id)
            continue
        
        # Rerank using dense embeddings
        reranked = reranker.rerank(
            question,
            mini_tables,
            valid_table_ids,
            top_k=config.stage2_candidates
        )
        
        reranked_results[query_id] = reranked
        
    except Exception as e:
        logger.error(f"Error processing query {query_id}: {e}")
        failed_queries.append(query_id)

logger.info(f"✅ Successfully processed {len(reranked_results)} queries")
if failed_queries:
    logger.warning(f"⚠️ Failed queries: {len(failed_queries)} - {failed_queries[:5]}...")

In [ ]:
def construct_mini_tables(qid, table_ids, top_rows_per_table, rowid_to_data, max_rows=5):
    """
    Construct mini-tables from top rows for given table IDs.
    
    Args:
        qid: Query ID
        table_ids: List of table IDs to process
        top_rows_per_table: Mapping of QID -> table_id -> row_ids
        rowid_to_data: Mapping of row_id -> row data
        max_rows: Maximum number of rows to include in mini-table
    
    Returns:
        mini_tables: List of mini-table text representations
        valid_table_ids: List of table IDs that had valid rows
    """
    mini_tables = []
    valid_table_ids = []
    
    for table_id in table_ids:
        # Get top rows for this table and query
        rows = top_rows_per_table.get(str(qid), {}).get(table_id, [])
        if not rows:
            continue
            
        # Convert row IDs to actual row text
        row_texts = []
        for rid in rows:
            if rid in rowid_to_data:
                row_texts.append(rowid_to_data[rid]["Row Data"])
        
        # Only include if we have valid rows
        if row_texts:
            # Take top N rows and join into mini-table
            mini_table_text = " ".join(row_texts[:max_rows])
            mini_tables.append(mini_table_text)
            valid_table_ids.append(table_id)
    
    return mini_tables, valid_table_ids

def rank_document_ids(query, documents, doc_ids):
    """
    Rank document IDs based on similarity to query.
    
    Args:
        query: Query text
        documents: List of document texts (mini-tables)
        doc_ids: List of document IDs (table IDs)
    
    Returns:
        ranked_ids: List of (doc_id, similarity_score) tuples, ranked by score
    """
    if len(documents) != len(doc_ids):
        raise ValueError("Number of documents must match doc_ids")
    
    # Get embeddings
    query_embedding = get_embedding(query)
    doc_embeddings = get_embeddings_batch(documents)
    
    # Calculate similarities
    similarities = cosine_similarity(query_embedding, doc_embeddings)
    
    # Create and sort (id, score) pairs
    id_sim_pairs = list(zip(doc_ids, similarities))
    ranked_ids = sorted(id_sim_pairs, key=lambda x: x[1], reverse=True)
    
    return [(doc_id, float(score)) for doc_id, score in ranked_ids]

print("Mini-table construction and ranking functions defined")

In [ ]:
# Save Stage 2 results using results manager
logger.info("💾 Saving Stage 2 results...")

# Save pickle format
pickle_path = results_manager.save_stage_results(
    reranked_results, 2, DATASET, config.get_path("stage2_output").parent
)

# Also save in JSONL format for compatibility
jsonl_path = config.base_dir / "results" / "stage2" / f"CRAFT_{DATASET.upper()}_2ndStage_Result_dense_reranking.jsonl"
jsonl_path.parent.mkdir(parents=True, exist_ok=True)

with open(jsonl_path, 'w') as f:
    for query_id, results in reranked_results.items():
        # Get query information for JSONL output
        if query_id in questions.index.astype(str):
            query_row = questions.loc[questions.index.astype(str) == query_id].iloc[0]
        else:
            query_idx = int(query_id) if query_id.isdigit() else 0
            query_row = questions.iloc[query_idx] if query_idx < len(questions) else {}
        
        question = query_row.get('question', query_row.get('text', ''))
        gold_table_id = query_row.get('gold_table_id', '')
        
        # Format top 100 results
        top_results = results[:config.stage2_candidates]
        
        output_data = {
            'query_id': query_id,
            'question': question,
            'gold_table_id': gold_table_id,
            'stage2_results': top_results,
            'num_candidates': len(top_results)
        }
        
        f.write(json.dumps(output_data, ensure_ascii=False) + '\n')

logger.info(f"✅ Stage 2 results saved to:")
logger.info(f"   Pickle: {pickle_path}")
logger.info(f"   JSONL: {jsonl_path}")
logger.info(f"🎯 Ready for Stage 3 processing")

In [ ]:
# Evaluate Stage 2 performance
if 'gold_table_id' in questions.columns:
    logger.info("📊 Computing Stage 2 evaluation metrics...")
    
    from craft_core import EvaluationMetrics
    metrics = EvaluationMetrics()
    
    # Prepare data for evaluation
    gold_answers = {}
    rankings = {}
    
    for query_id, results in reranked_results.items():
        # Get gold answer
        if query_id in questions.index.astype(str):
            query_row = questions.loc[questions.index.astype(str) == query_id].iloc[0]
        else:
            query_idx = int(query_id) if query_id.isdigit() else 0
            query_row = questions.iloc[query_idx] if query_idx < len(questions) else {}
        
        gold_table = query_row.get('gold_table_id', '')
        if gold_table:
            gold_answers[query_id] = gold_table
            rankings[query_id] = [tid for tid, _ in results]
    
    # Compute metrics
    recall_scores = metrics.compute_recall_at_k(rankings, gold_answers, [1, 10, 50])
    mrr_score = metrics.compute_mrr(rankings, gold_answers)
    
    logger.info("📈 Stage 2 Dense Reranking Results:")
    for k, score in recall_scores.items():
        logger.info(f"   Recall@{k}: {score:.2f}%")
    logger.info(f"   MRR: {mrr_score:.4f}")
    
    # Compare with Stage 1 if available
    logger.info(f"📋 Summary:")
    logger.info(f"   Queries processed: {len(reranked_results)}")
    logger.info(f"   Average candidates per query: {config.stage2_candidates}")
    logger.info(f"   Mini-table max rows: {config.mini_table_rows}")
    logger.info(f"   Model used: {model_name}")

logger.info("🎉 Stage 2 Dense Semantic Reranking completed!")